In [9]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "aircAruvnKk"

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)

for t in transcript[:5]:
    print(t)

FetchedTranscriptSnippet(text='This is a 3.', start=4.22, duration=1.18)
FetchedTranscriptSnippet(text="It's sloppily written and rendered at an extremely low resolution of 28x28 pixels,", start=6.06, duration=4.653)
FetchedTranscriptSnippet(text='but your brain has no trouble recognizing it as a 3.', start=10.713, duration=3.007)
FetchedTranscriptSnippet(text='And I want you to take a moment to appreciate how', start=14.34, duration=2.219)
FetchedTranscriptSnippet(text='crazy it is that brains can do this so effortlessly.', start=16.559, duration=2.401)


In [10]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "aircAruvnKk"

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id, languages=["hi"])

for t in transcript[:5]:
    print(t)

FetchedTranscriptSnippet(text='यह 3 है.', start=4.22, duration=1.18)
FetchedTranscriptSnippet(text='इसे 28x28 पिक्सल के बेहद कम रिज़ॉल्यूशन पर लापरवाही से लिखा और प्रस्तुत किया गया है, ', start=6.06, duration=4.019)
FetchedTranscriptSnippet(text='लेकिन आपके मस्तिष्क को इसे 3 के रूप में पहचानने में कोई परेशानी नहीं होती है।', start=10.079, duration=3.641)
FetchedTranscriptSnippet(text='और मैं चाहता हूं कि आप एक पल के लिए इसकी सराहना करें कि ', start=14.34, duration=2.33)
FetchedTranscriptSnippet(text='यह कितना अजीब है कि दिमाग इतनी आसानी से ऐसा कर सकता है।', start=16.67, duration=2.29)


In [2]:
import json
import os
from youtube_transcript_api import YouTubeTranscriptApi

os.makedirs("../transcripts", exist_ok=True)
os.makedirs("../output", exist_ok=True)

print("Setup complete.")

Setup complete.


In [3]:
VIDEOS = [
    {
        "id": "aircAruvnKk",
        "title": "3Blue1Brown — But what is a Neural Network?",
        "short_name": "3b1b_neural_networks",
        "language": "en"
    },
    {
        "id": "wjZofJX0v4M",
        "title": "3Blue1Brown — Transformers, the tech behind LLMs",
        "short_name": "3b1b_transformers",
        "language": "en"
    },
    {
        "id": "fHF22Wxuyw4",
        "title": "CampusX — What is Deep Learning? (Hindi)",
        "short_name": "campusx_deep_learning",
        "language": "hi"
    },
    {
        "id": "C6YtPJxNULA",
        "title": "CodeWithHarry — All About ML & Deep Learning (Hindi)",
        "short_name": "codewithharry_ml_dl",
        "language": "hi"
    }
]

print(f"Registered {len(VIDEOS)} videos.")

Registered 4 videos.


In [6]:
# Cell 3: Fetch and save transcripts

def fetch_and_save_transcript(video_info):
    """
    Fetches transcript for a video and saves it as a structured JSON file.
    
    Each entry in the JSON has:
    - text: the spoken words
    - start: start time in seconds
    - duration: how long the snippet lasts
    - timestamp: human-readable MM:SS format
    - video_title: which video this came from
    """
    ytt_api = YouTubeTranscriptApi()
    video_id = video_info["id"]
    lang = video_info["language"]
    short_name = video_info["short_name"]
    title = video_info["title"]
    
    save_path = f"../transcripts/{short_name}.json"
    
    # Skip re-downloading if already saved
    if os.path.exists(save_path):
        print(f"[SKIP] {title} — already saved.")
        return
    
    try:
        # Fetch in primary language; fall back to English auto-generated
        try:
            transcript = ytt_api.fetch(video_id, languages=[lang])
        except Exception:
            transcript = ytt_api.fetch(video_id, languages=["en"])
        
        # Structure each snippet
        structured = []
        for snippet in transcript:
            start_sec = snippet.start
            minutes = int(start_sec // 60)
            seconds = int(start_sec % 60)
            timestamp = f"{minutes:02d}:{seconds:02d}"
            
            structured.append({
                "text": snippet.text,
                "start": round(start_sec, 2),
                "duration": round(snippet.duration, 2),
                "timestamp": timestamp,
                "video_title": title
            })
        
        # Save to disk
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(structured, f, ensure_ascii=False, indent=2)
        
        print(f"[SAVED] {title} — {len(structured)} snippets → {save_path}")
    
    except Exception as e:
        print(f"[ERROR] {title} — {e}")


# Run for all 4 videos
for video in VIDEOS:
    fetch_and_save_transcript(video)

NameError: name 'VIDEOS' is not defined

In [7]:
# Cell 4: Load all transcripts into memory and inspect them

transcripts = {}

for video in VIDEOS:
    path = f"../transcripts/{video['short_name']}.json"
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    transcripts[video["short_name"]] = data
    print(f"Loaded: {video['title']} — {len(data)} snippets")

# Preview first 3 snippets from first video
print("\n--- Preview: 3b1b_neural_networks ---")
for snippet in transcripts["3b1b_neural_networks"][:3]:
    print(f"[{snippet['timestamp']}] {snippet['text']}")

NameError: name 'VIDEOS' is not defined

In [8]:
# Cell 6: Define the 5 Golden QA Pairs

# Each entry must have: question, answer, source (video + timestamp)
# These are manually crafted after reading the transcripts.
# They are designed to test RETRIEVAL PRECISION, not just recall.

golden_dataset = [
    {
        "id": "Q1",
        "question": "What is the role of activation functions in a neural network, and why can't a network without them learn complex patterns?",
        "answer": (
            "Activation functions introduce non-linearity into the network. "
            "Without them, stacking multiple layers would still produce only a linear transformation of the input — "
            "equivalent to a single-layer network. Non-linearity allows the network to learn complex, "
            "hierarchical representations such as edges, shapes, and objects in the case of images."
        ),
        "source": {
            "video": "3Blue1Brown — But what is a Neural Network?",
            "video_id": "aircAruvnKk",
            "timestamp": "~07:30",
            "note": "Section explaining neuron activations and sigmoid/ReLU functions"
        },
        "evaluation_notes": {
            "what_correct_retrieval_looks_like": "Chunk explaining non-linearity and why sigmoid/ReLU is applied",
            "what_wrong_retrieval_looks_like": "Chunk about network architecture or layer counts — correct topic but missing the 'why' of activation functions"
        }
    },
    {
        "id": "Q2",
        "question": "How does the attention mechanism in transformers allow the model to handle long-range dependencies in a sequence?",
        "answer": (
            "The attention mechanism computes a weighted relevance score between every pair of tokens in the sequence. "
            "This means a word at position 1 can directly influence a word at position 50 without needing to pass "
            "information through every intermediate step. Unlike RNNs which process tokens sequentially, "
            "attention processes all positions simultaneously, making long-range dependencies as easy to learn as short-range ones."
        ),
        "source": {
            "video": "3Blue1Brown — Transformers, the tech behind LLMs",
            "video_id": "wjZofJX0v4M",
            "timestamp": "~06:10",
            "note": "Section explaining query-key-value attention and why it replaces sequential processing"
        },
        "evaluation_notes": {
            "what_correct_retrieval_looks_like": "Chunk explaining Q-K-V attention computation and token interaction",
            "what_wrong_retrieval_looks_like": "Chunk about positional encoding — related topic, but doesn't answer the dependency question"
        }
    },
    {
        "id": "Q3",
        "question": "What distinguishes deep learning from classical machine learning in terms of feature engineering?",
        "answer": (
            "In classical machine learning, features (like edges, colour histograms, or frequency bands) must be manually "
            "designed by domain experts before being fed into the model. "
            "Deep learning eliminates this step — the network learns useful feature representations directly from raw data "
            "across its layers. Lower layers learn low-level features (edges, textures) and higher layers learn abstract "
            "concepts (faces, objects, sentiment)."
        ),
        "source": {
            "video": "CampusX — What is Deep Learning? (Hindi)",
            "video_id": "fHF22Wxuyw4",
            "timestamp": "~05:00",
            "note": "Section comparing traditional ML pipeline vs deep learning pipeline"
        },
        "evaluation_notes": {
            "what_correct_retrieval_looks_like": "Chunk contrasting manual feature engineering with learned representations",
            "what_wrong_retrieval_looks_like": "Chunk listing deep learning applications — topically adjacent but doesn't address the feature engineering distinction"
        }
    },
    {
        "id": "Q4",
        "question": "Why do neural networks require large amounts of labelled training data compared to traditional ML models?",
        "answer": (
            "Neural networks have millions (or billions) of parameters that need to be tuned. "
            "With few examples, the network overfits — it memorises the training data rather than generalising. "
            "Traditional ML models use handcrafted features and simpler decision boundaries, "
            "so they can work well with far fewer examples. The expressiveness of deep networks is both their "
            "strength and the reason they are data-hungry."
        ),
        "source": {
            "video": "CodeWithHarry — All About ML & Deep Learning (Hindi)",
            "video_id": "C6YtPJxNULA",
            "timestamp": "~08:20",
            "note": "Section discussing limitations of deep learning and data requirements"
        },
        "evaluation_notes": {
            "what_correct_retrieval_looks_like": "Chunk about overfitting, parameter count, or data requirements for DL",
            "what_wrong_retrieval_looks_like": "Chunk about hardware requirements (GPUs) — often mentioned alongside data, but answers a different question"
        }
    },
    {
        "id": "Q5",
        "question": "How does backpropagation use the chain rule to update weights in a neural network?",
        "answer": (
            "Backpropagation computes the gradient of the loss with respect to each weight by applying the chain rule "
            "of calculus backward through the network. Starting from the output layer, the error signal is propagated "
            "back layer by layer — at each step, the gradient is the product of the local derivative and the gradient "
            "flowing from the layer ahead. These gradients tell the optimizer (e.g., gradient descent) how much and "
            "in which direction to adjust each weight."
        ),
        "source": {
            "video": "3Blue1Brown — But what is a Neural Network?",
            "video_id": "aircAruvnKk",
            "timestamp": "~15:00",
            "note": "Section explaining the gradient descent and backpropagation intuition"
        },
        "evaluation_notes": {
            "what_correct_retrieval_looks_like": "Chunk explaining chain rule application, gradient flow backward through layers",
            "what_wrong_retrieval_looks_like": "Chunk explaining forward pass or loss functions — adjacent but doesn't describe the backward update mechanism"
        }
    }
]

print(f"Golden dataset contains {len(golden_dataset)} QA pairs.")
for qa in golden_dataset:
    print(f"  {qa['id']}: {qa['question'][:60]}...")

Golden dataset contains 5 QA pairs.
  Q1: What is the role of activation functions in a neural network...
  Q2: How does the attention mechanism in transformers allow the m...
  Q3: What distinguishes deep learning from classical machine lear...
  Q4: Why do neural networks require large amounts of labelled tra...
  Q5: How does backpropagation use the chain rule to update weight...


In [ ]:
# Cell 7: Methodology Note (will be included in final output)

methodology = {
    "question_selection_criteria": [
        "Prioritised questions that require reasoning about mechanisms, not just definitions — a RAG system that retrieves shallow definitional chunks should score lower",
        "Ensured coverage across all 4 videos to test retrieval across a diverse corpus",
        "Selected concepts that are semantically adjacent to other concepts (e.g., activation functions vs layer architecture) — this creates meaningful hard negatives for retrieval evaluation",
        "Avoided yes/no questions; all 5 require multi-sentence answers grounded in specific transcript sections"
    ],
    "extraction_process": [
        "Pulled all 4 transcripts programmatically using youtube-transcript-api with language-appropriate fetching (en for 3B1B, hi for CampusX and CodeWithHarry)",
        "Converted raw start times (seconds) to human-readable MM:SS timestamps",
        "Used keyword search across transcripts to locate concept-dense segments",
        "Manually reviewed surrounding context windows (~3 snippets before/after) to verify the answer was fully grounded in that section"
    ],
    "what_these_questions_test": [
        "Retrieval precision — can the system find the specific chunk containing the mechanistic explanation, not just the topic area?",
        "Semantic grounding — answers require the retrieved chunk to contain reasoning, not just keywords",
        "Cross-video retrieval — questions span all 4 videos, so a system must retrieve from the right source",
        "Hard negative resistance — each question has a plausible wrong retrieval documented; a good retrieval system should rank the correct chunk higher"
    ],
    "what_wrong_retrieval_looks_like": [
        "Returning a chunk that mentions the keyword but lacks the explanatory content (e.g., 'attention' appears in many chunks, but only a few explain the Q-K-V mechanism)",
        "Returning a chunk from the wrong video on the same topic",
        "Returning a definitional chunk when the question asks about a mechanism or comparison"
    ]
}

print("Methodology note prepared.")